<a href="https://colab.research.google.com/github/elfantasies/AI/blob/main/0704_Colab_LINE_Bot_with_GEMINI_Tooluse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://sequential-dispensational-tripp.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://sequential-dispensational-tripp.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [8]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。 該校地處桃竹苗區域核心，緊鄰新竹科學園區與新竹工業區，交通便利，坐擁豐富的產業資源。

**歷史沿革**
明新科技大學創立於1966年3月，最初以「明新工業專科學校」立案，設有機械、土木、工業管理等科系。 隨著時代發展，學校逐步擴充科系，並於1993年更名為「私立明新工商專科學校」。 1997年7月，奉教育部核准改制為「明新技術學院」，並附設專科部。 最終在2002年8月（民國九十一年八月）奉教育部核准升格為「明新科技大學」。2018年12月，更名為「明新學校財團法人明新科技大學」。

**學術單位與特色**
明新科技大學目前設有六個學院，包括半導體學院、工程學院、管理學院、民生學院、人文與設計學院以及共同教育學院。這些學院涵蓋了20個學系、2個學位學程（含1個博士學位學程）及11個碩士班。學校以「深耕在地、放眼國際」為願景，目標是培養具備專業技能、團隊精神、人文素養與宏觀服務的科技人才。

明新科大以「產業大學」為辦學定位，強調產學鏈結與實務人才培育。學校在2021年整合電機、電子、化學工程與材料科技、光電工程等學系，成立了半導體學院，以因應台灣半導體產業的發展需求。在半導體領域，明新科大透過「課程、類產線實習、證照考試一條龍」的育才模式，培養業界搶手人才，並與台積電等企業有實習合作計畫。

此外，學校也積極推動AI教育與國際化，將程式設計與人工智慧概論列為大一必修課程，並將AI技術融入各領域教學，以培育AI時代的專業人才。 明新科大也致力於推動「智慧生活」的創新服務，建置「永續智慧商務」教學與實習場域，發展智慧零售、智慧金融、智慧製造與智慧商業等實驗室。

其畢業校友已逾十萬人，在各行各業中擔任中堅份子。 在企業界享有良好聲譽，曾獲得《遠見》雜誌「企業最愛公私立技職科大調查」中的多項肯定，並在半導體產業中最受企業青睞的畢業生排名中名列前茅。


In [9]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [10]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 04:50:20] "POST / HTTP/1.1" 400 -


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[]}


INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 04:50:28] "POST / HTTP/1.1" 400 -


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[]}


INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 04:51:05] "POST / HTTP/1.1" 400 -


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[]}
